# Day 2 · 한국어 회의 음성을 근거가 남는 결과로 바꾸기

화면을 따라 실행하되, 결과를 자동 게시하지 않습니다. 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
# 최초 1회 설치. 이미 설치했다면 빠르게 완료됩니다.
%pip install -q -r ../../requirements-day1.txt
# STT 실습을 실제 음성으로 실행할 때만 다음 줄의 주석을 해제합니다.
# %pip install -q -r ../../requirements-stt-optional.txt

In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": str(ROOT), "python": sys.version.split()[0]})

## 1. 오디오와 정답 전사문을 먼저 확인합니다

`RUN_STT_LIVE=False`가 기본값입니다. 수업 흐름은 fixture로 먼저 검증한 뒤, 모델이 준비된 컴퓨터만 실제 STT를 실행합니다.

In [ ]:
import wave

audio_path = ROOT / "data/demo_meeting.wav"
transcript_path = ROOT / "data/demo_meeting_transcript.txt"
with wave.open(str(audio_path), "rb") as wav:
    audio_meta = {
        "seconds": round(wav.getnframes() / wav.getframerate(), 2),
        "sample_rate": wav.getframerate(),
        "channels": wav.getnchannels(),
    }
print(audio_meta)
print(transcript_path.read_text(encoding="utf-8")[:500])

## 2. 발화 단위로 나눈 뒤 chunk와 evidence ID를 보존합니다

In [ ]:
from src.course_services.meeting_service import prepare_transcript_for_summary

transcript = transcript_path.read_text(encoding="utf-8")
prepared = prepare_transcript_for_summary(transcript, max_chars=500)
print(json.dumps({
    "status": prepared["status"],
    "segment_count": len(prepared["segments"]),
    "chunks": [c["segment_ids"] for c in prepared["chunks"]],
}, ensure_ascii=False, indent=2))

## 3. 선택적으로 faster-whisper를 실행하고 품질 gate를 확인합니다

In [ ]:
from src.meeting_demo import run_demo

RUN_STT_LIVE = False
if RUN_STT_LIVE:
    stt_result = run_demo(
        audio_path=audio_path,
        transcript_path=transcript_path,
        output_dir=ROOT / "output/day2-stt-lab",
        model_size="small",
        device="cpu",
        compute_type="int8",
        local_files_only=True,
    )
    print(json.dumps(stt_result["quality_gate"], ensure_ascii=False, indent=2))
else:
    print("STT_LIVE_SKIPPED: fixture 기반 chunk·schema 실습을 계속합니다.")

## 완료 확인

- Day 2 결과 JSON을 확인했습니다.
- 실패 경로가 traceback 대신 `error_code`로 남는지 확인했습니다.
- 외부 쓰기와 자동 메일이 발생하지 않았음을 확인했습니다.
- 변경한 코드는 diff와 test 결과를 사람이 검토합니다.